In [11]:
# importing
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [12]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:7]}")
else:
    print("OpenRouter API key not set")


OpenAI API Key exists and begins sk-proj-
OpenRouter API key exists and begins sk-or-v


In [13]:
# Clients: OpenAI (GPT) and OpenRouter (Claude)
openai_client = OpenAI(api_key=openai_api_key)

openrouter_url = "https://openrouter.ai/api/v1"
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

GPT_MODEL = "gpt-4.1-mini"
CLAUDE_MODEL = "anthropic/claude-3.5-sonnet"


In [14]:
# setting system prompt
system_message = """
You are a helpful tutor.
Answer the student's question concisely and clearly based on the information you have.
Keep your answer under 400 tokens. If you don't know the answer, say so.
Respond in markdown format.
"""


In [15]:
# Chat with model choice and streaming (chunks/delta)
def get_client_and_model(model_choice: str):
    """Return (client, model_id) for the chosen UI option."""
    if model_choice == "Claude":
        return openrouter_client, CLAUDE_MODEL
    return openai_client, GPT_MODEL


def chat(message, history, model_choice):
    client, model_id = get_client_and_model(model_choice)
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = client.chat.completions.create(model=model_id, messages=messages, stream=True)
    partial = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            partial += chunk.choices[0].delta.content
            yield partial


In [16]:
# Launch chat interface with model dropdown and streaming
model_dropdown = gr.Dropdown(
    choices=["GPT", "Claude"],
    value="GPT",
    label="Model",
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_dropdown],
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
